# Trust Game: canonical events to first-level FEAT

**Author:** Smith Lab  
**Updated:** 2026-08-18  
**License:** MIT

This notebook calls the repository production EV generator and model-1 L1 worker for Trust runs 1 and 2. It does not reconstruct behavior or define a notebook-only model.

## What you will learn

1. Inspect canonical Trust timing, zero investments, and misses.
2. Generate FSL EVs without changing onset/duration.
3. Build a transparent teaching nuisance file.
4. Run and inspect the established 10-EV/18-contrast activation model.


## 1. Load pinned software and locate the repository

In [ ]:
import module
await module.load('fsl/6.0.7.22')
await module.list()

In [ ]:
%pip install -q pandas nibabel matplotlib watermark
from pathlib import Path
import os, subprocess
import pandas as pd
import nibabel as nib
import matplotlib.pyplot as plt
from IPython.display import IFrame, Image, display

In [ ]:
SUBJECT='10317'; SESSION='01'; TASK='trust'; RUNS=('1','2')
WORKSPACE=Path.home()/'trust_teaching'; BIDS_DIR=WORKSPACE/'ds005123'; FMRIPREP_DIR=WORKSPACE/'derivatives'; FSL_DIR=WORKSPACE/'fsl'; TEACHING_CONFOUNDS_DIR=WORKSPACE/'teaching_confounds'
REPO=next((p for p in [Path.cwd(),*Path.cwd().parents] if (p/'code'/'L1stats.sh').is_file()),None)
if REPO is None: raise FileNotFoundError('Run inside a clone of rf1-sra-trust.')
for path in (FSL_DIR,TEACHING_CONFOUNDS_DIR): path.mkdir(parents=True,exist_ok=True)
workflow_env={**os.environ,'BIDS_ROOT':str(BIDS_DIR),'FMRIPREP_ROOT':str(FMRIPREP_DIR),'CONFOUNDS_ROOT':str(TEACHING_CONFOUNDS_DIR),'FSL_DERIVATIVES_ROOT':str(FSL_DIR)}
print(REPO)

## 2. Inspect authoritative BIDS timing

A zero investment has a valid `choice_*` but no invented outcome. The downstream converter copies onset and duration exactly.

In [ ]:
event_files={}
for run in RUNS:
    path=BIDS_DIR/f'sub-{SUBJECT}'/f'ses-{SESSION}'/'func'/f'sub-{SUBJECT}_ses-{SESSION}_task-{TASK}_run-{run}_events.tsv'
    if not path.is_file(): raise FileNotFoundError(path)
    event_files[run]=path; frame=pd.read_csv(path,sep='\t')
    print(f'run-{run}: {len(frame)} event rows'); display(frame[['onset','duration','trial_type','trust_value']].head(12)); display(frame.trial_type.value_counts())

## 3. Generate FSL EVs with the production script

In [ ]:
for run in RUNS:
    subprocess.run(['bash',str(REPO/'code'/'gen3colfiles.sh'),'--subject',SUBJECT,'--session',SESSION,'--run',run,'--overwrite'],env=workflow_env,check=True)
    ev_dir=FSL_DIR/'EVfiles'/f'sub-{SUBJECT}'/f'ses-{SESSION}'/'trust'
    print(f'run-{run}')
    for ev in sorted(ev_dir.glob(f'run-{run}_*.txt')): print(f'  {ev.name}: {len(pd.read_csv(ev,sep=r"\s+",header=None))} events')

## 4. Create simplified teaching confounds

> Production uses Linux2 TEDANA-enhanced confounds. This public exercise uses fMRIPrep cosine/non-steady-state terms, six motion parameters, six aCompCor components, and framewise displacement.

In [ ]:
def one(paths,label):
    paths=list(paths)
    if len(paths)!=1: raise RuntimeError(f'Expected one {label}: {paths}')
    return paths[0]
confound_files={}; bold_files={}; func=FMRIPREP_DIR/f'sub-{SUBJECT}'/f'ses-{SESSION}'/'func'
for run in RUNS:
    source=one(func.glob(f'*task-{TASK}_run-{run}*desc-confounds_timeseries.tsv'),f'run-{run} confounds')
    frame=pd.read_csv(source,sep='\t'); motion=['trans_x','trans_y','trans_z','rot_x','rot_y','rot_z']; columns=[x for x in frame if x.startswith('cosine') or x.startswith('non_steady_state_outlier')]+motion+[x for x in frame if x.startswith('a_comp_cor_')][:6]+['framewise_displacement']; columns=list(dict.fromkeys(x for x in columns if x in frame))
    selected=frame[columns].apply(pd.to_numeric,errors='raise').fillna(0); outdir=TEACHING_CONFOUNDS_DIR/f'sub-{SUBJECT}'; outdir.mkdir(parents=True,exist_ok=True); stem=f'sub-{SUBJECT}_ses-{SESSION}_task-{TASK}_run-{run}'; out=outdir/f'{stem}_desc-TeachingFmriprepConfounds.tsv'; selected.to_csv(out,sep='\t',index=False,header=False); confound_files[run]=out
    bold_files[run]=one((p for p in func.glob(f'*task-{TASK}_run-{run}*space-MNI152NLin6Asym*desc-preproc_bold.nii.gz') if 'echo-' not in p.name),f'run-{run} BOLD')
    print(f'run-{run}: {selected.shape} -> {out}')

## 5. Render and run both L1 activation models

PPI/nPPI are outside this introductory notebook.

In [ ]:
for run in RUNS:
    command=['bash',str(REPO/'code'/'L1stats.sh'),SUBJECT,run,'0','--session',SESSION,'--bold',str(bold_files[run]),'--confounds',str(confound_files[run]),'--render-only']
    subprocess.run(command,env=workflow_env,check=True)
    rendered=FSL_DIR/f'sub-{SUBJECT}'/f'ses-{SESSION}'/f'L1_sub-{SUBJECT}_task-trust_ses-{SESSION}_model-1_type-act_run-{run}.fsf'
    print(f'run-{run}: {rendered}'); print('\n'.join(line for line in rendered.read_text().splitlines() if any(x in line for x in ('outputdir','npts','shape10','feat_files(1)','confoundev_files(1)'))))

In [ ]:
OVERWRITE_INCOMPLETE=False
for run in RUNS:
    command=['bash',str(REPO/'code'/'L1stats.sh'),SUBJECT,run,'0','--session',SESSION,'--bold',str(bold_files[run]),'--confounds',str(confound_files[run])]
    if OVERWRITE_INCOMPLETE: command.append('--overwrite')
    subprocess.run(command,env=workflow_env,check=True)

## 6. Inspect FEAT and reciprocation > defection

Contrast 10 is the established `rec-def` outcome contrast across partners. Inspect the complete FEAT reports before interpreting images.

In [ ]:
feat_dirs={}
for run in RUNS:
    feat=FSL_DIR/f'sub-{SUBJECT}'/f'ses-{SESSION}'/f'L1_task-trust_ses-{SESSION}_model-1_type-act_run-{run}_sm-5.feat'; feat_dirs[run]=feat
    if not (feat/'design.png').is_file() or not (feat/'report.html').is_file(): raise FileNotFoundError(feat)
    display(Image(filename=str(feat/'design.png'))); display(IFrame(src=str(feat/'report.html'),width='100%',height=600))
fig,axes=plt.subplots(1,2,figsize=(12,5))
for ax,run in zip(axes,RUNS):
    img=nib.load(feat_dirs[run]/'stats'/'zstat10.nii.gz'); data=img.get_fdata(); z=data.shape[2]//2; shown=ax.imshow(data[:,:,z].T,cmap='coolwarm',origin='lower',vmin=-5,vmax=5); ax.set_title(f'run-{run}: rec > defect'); ax.axis('off')
fig.colorbar(shown,ax=axes,shrink=.7,label='Z'); plt.show()

## 7. Handoff

Notebook 03 combines these exact run-1 and run-2 outputs with the production fixed-effects L2 worker.

In [ ]:
%load_ext watermark
%watermark
%watermark --iversions
await module.list()